# 🔬 EyeQ Innovate — Complete Training Pipeline
### Multi-Label Retinal Disease Classification | 10 Diseases | ConvNeXt-Base | 2×T4 GPU

**Diseases Classified:**
| # | Code | Disease |
|---|------|---------|
| 0 | DR   | Diabetic Retinopathy |
| 1 | G    | Glaucoma |
| 2 | AMD  | Age-Related Macular Degeneration |
| 3 | C    | Cataract |
| 4 | M    | Pathological Myopia |
| 5 | HR   | Hypertensive Retinopathy |
| 6 | DME  | Diabetic Macular Edema |
| 7 | P    | Papilledema |
| 8 | CSR  | Central Serous Retinopathy |
| 9 | RVO  | Retinal Vein Occlusion |


## ⚙️ Stage 0 — Install Dependencies

In [ ]:
%%capture
!pip install timm albumentations grad-cam iterative-stratification imagehash reportlab openpyxl --quiet
print('✅ All packages installed')

## 📦 Stage 1 — Imports & Config

In [ ]:
import os, gc, cv2, json, warnings, hashlib, shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from PIL import Image
from tqdm.notebook import tqdm
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.cuda.amp import GradScaler, autocast
import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.metrics import f1_score, roc_auc_score, precision_score, recall_score
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold

# ─── GPU Setup ───────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
n_gpus = torch.cuda.device_count()
print(f'🖥️  Device : {device}')
print(f'🔢 GPUs   : {n_gpus}')
for i in range(n_gpus):
    print(f'   GPU {i}: {torch.cuda.get_device_name(i)} — '
          f'{torch.cuda.get_device_properties(i).total_memory / 1e9:.1f} GB')

# ─── Config ──────────────────────────────────────────────────
class CFG:
    # Paths
    ROOT          = Path('/kaggle/input/datasets')
    OUTPUT_DIR    = Path('/kaggle/working/outputs')
    MODEL_DIR     = Path('/kaggle/working/models')

    # Diseases
    DISEASES      = ['DR','G','AMD','C','M','HR','DME','P','CSR','RVO']
    N_CLASSES     = 10

    # Training
    MODEL_NAME    = 'convnext_base'
    IMG_SIZE      = 512
    BATCH_SIZE    = 32      # dynamically adjusted below
    NUM_WORKERS   = 2       # safer for Kaggle multiprocessing
    EPOCHS        = 2
    LIMIT_FOLDS   = 1       # train 1 fold for exact 2-epoch run (set to N_FOLDS for all)
    LR            = 2e-4    # scaled for batch 32
    WEIGHT_DECAY  = 1e-4
    N_FOLDS       = 5
    SEED          = 42

    # Class weights for imbalanced diseases
    CLASS_WEIGHTS = [1.0, 1.2, 1.1, 1.0, 1.0, 1.5, 2.0, 3.0, 2.5, 3.0]

    # Thresholds (tuned per class on validation)
    THRESHOLDS    = [0.5] * 10  # will be tuned later

CFG.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CFG.MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Dynamically scale batch size based on available GPUs to prevent OOM
if torch.cuda.is_available():
    n_gpus_available = torch.cuda.device_count()
    if n_gpus_available >= 2:
        CFG.BATCH_SIZE = 32
    else:
        CFG.BATCH_SIZE = 16  # safe for single T4/L4 GPU
else:
    CFG.BATCH_SIZE = 4       # safe fallback for CPU
print(f'⚙️ Batch size set to {CFG.BATCH_SIZE} based on accelerator context')

def seed_everything(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)

seed_everything(CFG.SEED)
print('\n✅ Config ready')

## 🗂️ Stage 2 — Dataset Paths & Label Mapping

In [ ]:
# ─── Dataset Paths ───────────────────────────────────────────
PATHS = {
    # RFMiD (andrewmvd/retinal-disease-classification)
    'rfmid_train_img' : CFG.ROOT / 'andrewmvd/retinal-disease-classification/Training_Set/Training_Set/Training',
    'rfmid_train_csv' : CFG.ROOT / 'andrewmvd/retinal-disease-classification/Training_Set/Training_Set/RFMiD_Training_Labels.csv',
    'rfmid_val_img'   : CFG.ROOT / 'andrewmvd/retinal-disease-classification/Evaluation_Set/Evaluation_Set/Validation',
    'rfmid_val_csv'   : CFG.ROOT / 'andrewmvd/retinal-disease-classification/Evaluation_Set/Evaluation_Set/RFMiD_Validation_Labels.csv',
    'rfmid_test_img'  : CFG.ROOT / 'andrewmvd/retinal-disease-classification/Test_Set/Test_Set/Test',
    'rfmid_test_csv'  : CFG.ROOT / 'andrewmvd/retinal-disease-classification/Test_Set/Test_Set/RFMiD_Testing_Labels.csv',

    # RFMiD 2.0
    'rfmid2_train_img': CFG.ROOT / 'sanjaykrish2006/rfmid2-0/Training_set',
    'rfmid2_val_img'  : CFG.ROOT / 'sanjaykrish2006/rfmid2-0/Validation_set',
    'rfmid2_test_img' : CFG.ROOT / 'sanjaykrish2006/rfmid2-0/Test_set',

    # ODIR-5K
    'odir_train_img'  : CFG.ROOT / 'andrewmvd/ocular-disease-recognition-odir5k/preprocessed_images',
    'odir_csv'        : CFG.ROOT / 'andrewmvd/ocular-disease-recognition-odir5k/ODIR-5K/ODIR-5K/data.xlsx',

    # DDR (DR grading only)
    'ddr_dir'         : CFG.ROOT / 'mariaherrerot/ddrdataset/DR_grading/DR_grading',

    # Fundus1000
    'fundus1000'      : CFG.ROOT / 'linchundan/fundusimage1000/1000images/1000images',

    # Papilledema
    'papilledema'     : CFG.ROOT / 'shashwatwork/identification-of-pseudopapilledema/Papilledema',
    'normal_pap'      : CFG.ROOT / 'shashwatwork/identification-of-pseudopapilledema/Normal',

    # Hypertensive Retinopathy
    'hr_train'        : CFG.ROOT / 'harshwardhanfartale/hypertension-and-hypertensive-retinopathy-dataset/2-Hypertensive Retinopathy Classification/2-Hypertensive Retinopathy Classification/1-Images/1-Training Set',
    'hr_gt'           : CFG.ROOT / 'harshwardhanfartale/hypertension-and-hypertensive-retinopathy-dataset/2-Hypertensive Retinopathy Classification/2-Hypertensive Retinopathy Classification/2-Groundtruths',
}

print('📂 Dataset paths configured')

# ─── Verify paths exist ───────────────────────────────────────
print('\n🔍 Path verification:')
for name, path in PATHS.items():
    exists = Path(path).exists()
    status = '✅' if exists else '❌ MISSING'
    print(f'   {status}  {name}')

## 🏷️ Stage 3 — Build Unified Master Label CSV

In [ ]:
records = []  # list of dicts: {image_path, DR, G, AMD, C, M, HR, DME, P, CSR, RVO}

def empty_label():
    return {d: 0 for d in CFG.DISEASES}

# ────────────────────────────────────────────────────────────
# 1. RFMiD (andrewmvd version) — has structured CSV
# ────────────────────────────────────────────────────────────
print('Processing RFMiD...')
for split, img_dir, csv_path in [
    ('train', PATHS['rfmid_train_img'], PATHS['rfmid_train_csv']),
    ('val',   PATHS['rfmid_val_img'],   PATHS['rfmid_val_csv']),
    ('test',  PATHS['rfmid_test_img'],  PATHS['rfmid_test_csv']),
]:
    if not Path(csv_path).exists():
        # Try finding any CSV in parent dir
        csvs = list(Path(img_dir).parent.glob('*.csv'))
        if csvs: csv_path = csvs[0]
        else: print(f'  ⚠️  No CSV for RFMiD {split}, skipping'); continue

    df = pd.read_csv(csv_path)
    df.columns = [c.strip() for c in df.columns]

    # RFMiD column mappings (corrected for Glaucoma and expanded to maximize disease capture)
    col_map = {
        'DR':  ['DR','Diabetic Retinopathy','diabetic_retinopathy'],
        'G':   ['G','Glaucoma','GLAUCOMA','glaucoma','ODC'],
        'AMD': ['AMD','ARMD','Age-related Macular Degeneration','DN'],
        'M':   ['MYA','Myopia','myopia'],
        'HR':  ['AHR','Hypertensive Retinopathy','hypertensive retinopathy'],
        'DME': ['DME','Diabetic Macular Edema'],
        'CSR': ['CSCR','CSR','Central Serous Retinopathy','CRS'],
        'RVO': ['BRVO','CRVO','RVO','OCT'],
    }

    for _, row in df.iterrows():
        img_id = str(row.get('ID', row.get('id', row.iloc[0]))).strip()
        # Try multiple extensions
        img_path = None
        for ext in ['.png','.jpg','.jpeg','.PNG','.JPG']:
            p = Path(img_dir) / f'{img_id}{ext}'
            if p.exists(): img_path = str(p); break
        if img_path is None: continue

        label = empty_label()
        for disease, candidates in col_map.items():
            for c in candidates:
                if c in df.columns and row[c] == 1:
                    label[disease] = 1; break
        label['image_path'] = img_path
        records.append(label)

print(f'  → {len(records)} records after RFMiD')

# ────────────────────────────────────────────────────────────
# 2. ODIR-5K — xlsx with diagnostic keywords
# ────────────────────────────────────────────────────────────
print('Processing ODIR-5K...')
odir_before = len(records)
try:
    odir_df = pd.read_excel(PATHS['odir_csv'])
    odir_df.columns = [c.strip() for c in odir_df.columns]

    keyword_map = {
        'DR':  ['diabetic retinopathy','proliferative retinopathy','DR'],
        'G':   ['glaucoma'],
        'AMD': ['age-related macular degeneration','AMD','macular degeneration'],
        'C':   ['cataract'],
        'M':   ['myopia','pathologic myopia','pathological myopia'],
        'HR':  ['hypertensive retinopathy','hypertension'],
    }

    img_dir = PATHS['odir_train_img']
    for _, row in odir_df.iterrows():
        for eye, diag_col in [('Left', 'Left-Diagnostic Keywords'), ('Right', 'Right-Diagnostic Keywords')]:
            fname_col = f'{eye}-Fundus'
            if fname_col not in odir_df.columns: continue
            img_file = str(row.get(fname_col,'')).strip()
            if not img_file: continue
            img_path = Path(img_dir) / img_file
            if not img_path.exists(): continue

            diag = str(row.get(diag_col,'')).lower()
            if 'normal' in diag and not any(k in diag for keys in keyword_map.values() for k in keys):
                continue  # skip pure normals to save balance

            label = empty_label()
            label['image_path'] = str(img_path)
            for disease, keywords in keyword_map.items():
                if any(kw in diag for kw in keywords):
                    label[disease] = 1
            if sum(label[d] for d in CFG.DISEASES) > 0:
                records.append(label)
except Exception as e:
    print(f'  ⚠️  ODIR error: {e}')
print(f'  → {len(records) - odir_before} records from ODIR')

# ────────────────────────────────────────────────────────────
# 3. DDR — DR grading (0=No DR, 1-4 = DR grades)
# ────────────────────────────────────────────────────────────
print('Processing DDR...')
ddr_before = len(records)
for split in ['train','valid','test']:
    img_dir = Path(PATHS['ddr_dir']) / split / 'image'
    lbl_dir = Path(PATHS['ddr_dir']) / split / 'label'
    if not img_dir.exists(): continue
    lbl_file = lbl_dir.parent / f'{split}.txt' if not lbl_dir.exists() else None

    # Try paired label file
    label_dict = {}
    for txt in [Path(PATHS['ddr_dir'])/f'{split}.txt',
                 Path(PATHS['ddr_dir'])/split/f'{split}.txt']:
        if txt.exists():
            try:
                for line in txt.read_text(encoding='utf-8', errors='ignore').strip().split('\n'):
                    parts = line.strip().split()
                    if len(parts) >= 2:
                        label_dict[parts[0]] = int(parts[1])
            except Exception as e:
                print(f'  ⚠️ Error reading label text file: {e}')
            break

    for img_path in img_dir.glob('*.*'):
        grade = label_dict.get(img_path.name, -1)
        if grade == 0: continue  # No DR → skip
        label = empty_label()
        label['image_path'] = str(img_path)
        label['DR'] = 1 if grade >= 1 else 0
        label['DME'] = 1 if grade >= 3 else 0  # severe DR often has DME
        if label['DR'] == 1:
            records.append(label)
print(f'  → {len(records) - ddr_before} records from DDR')

# ────────────────────────────────────────────────────────────
# 4. Fundus1000 — folder name = disease
# ────────────────────────────────────────────────────────────
print('Processing Fundus1000...')
fundus_before = len(records)
FUNDUS_MAP = {
    '0.3.DR1':                      {'DR':1},
    '1.0.DR2':                      {'DR':1},
    '1.1.DR3':                      {'DR':1},
    '2.0.BRVO':                     {'RVO':1},
    '2.1.CRVO':                     {'RVO':1},
    '5.0.CSCR':                     {'CSR':1},
    '9.Pathological myopia':        {'M':1},
    '10.0.Possible glaucoma':       {'G':1},
    '11.Severe hypertensive retinopathy': {'HR':1},
    '12.Disc swelling and elevation':     {'P':1},
    '6.Maculopathy':                {'AMD':1},
    '0.0.Normal':                   {},
}
fundus_root = Path(PATHS['fundus1000'])
for folder, disease_labels in FUNDUS_MAP.items():
    folder_path = fundus_root / folder
    if not folder_path.exists(): continue
    if not disease_labels: continue  # skip normals
    for img_path in folder_path.glob('*.*'):
        if img_path.suffix.lower() not in ['.jpg','.jpeg','.png']: continue
        label = empty_label()
        label['image_path'] = str(img_path)
        label.update(disease_labels)
        records.append(label)
print(f'  → {len(records) - fundus_before} records from Fundus1000')

# ────────────────────────────────────────────────────────────
# 5. Papilledema Dataset
# ────────────────────────────────────────────────────────────
print('Processing Papilledema...')
pap_before = len(records)
for img_path in Path(PATHS['papilledema']).glob('*.*'):
    if img_path.suffix.lower() not in ['.jpg','.jpeg','.png']: continue
    label = empty_label()
    label['image_path'] = str(img_path)
    label['P'] = 1
    records.append(label)
print(f'  → {len(records) - pap_before} records from Papilledema')

# ────────────────────────────────────────────────────────────
# 6. Hypertensive Retinopathy Dataset
# ────────────────────────────────────────────────────────────
print('Processing HR dataset...')
hr_before = len(records)
hr_dir = Path(PATHS['hr_train'])
if hr_dir.exists():
    for img_path in hr_dir.rglob('*.*'):
        if img_path.suffix.lower() not in ['.jpg','.jpeg','.png']: continue
        label = empty_label()
        label['image_path'] = str(img_path)
        label['HR'] = 1
        records.append(label)
print(f'  → {len(records) - hr_before} records from HR dataset')

# ────────────────────────────────────────────────────────────
# Build Master DataFrame
# ────────────────────────────────────────────────────────────
master_df = pd.DataFrame(records)[['image_path'] + CFG.DISEASES]
master_df = master_df.drop_duplicates(subset=['image_path']).reset_index(drop=True)

# Remove rows with no disease label (all zeros = normal, keep only ~10% for balance)
disease_sum = master_df[CFG.DISEASES].sum(axis=1)
diseased = master_df[disease_sum > 0]
normals  = master_df[disease_sum == 0].sample(frac=0.1, random_state=42)
master_df = pd.concat([diseased, normals]).reset_index(drop=True)

print(f'\n📊 Master CSV: {len(master_df)} total samples')
print('\nClass distribution:')
for d in CFG.DISEASES:
    n = master_df[d].sum()
    print(f'  {d:5s}: {n:5d} ({100*n/len(master_df):.1f}%)')

# Save
master_df.to_csv(CFG.OUTPUT_DIR / 'master_labels.csv', index=False)
print('\n✅ master_labels.csv saved')

## 🧹 Stage 4 — Data Cleaning (Dedup & Corrupt Check)

In [ ]:
import imagehash

print('🔍 Removing corrupted images...')
valid_paths = []
for path in tqdm(master_df['image_path'], desc='Checking images'):
    try:
        img = Image.open(path)
        img.verify()
        valid_paths.append(path)
    except:
        pass

before = len(master_df)
master_df = master_df[master_df['image_path'].isin(valid_paths)].reset_index(drop=True)
print(f'  Removed {before - len(master_df)} corrupted images')

print('\n🔍 Removing duplicate images (perceptual hash)...')
hashes = {}
duplicates = []
for path in tqdm(master_df['image_path'], desc='Hashing'):
    try:
        h = str(imagehash.phash(Image.open(path)))
        if h in hashes:
            duplicates.append(path)
        else:
            hashes[h] = path
    except:
        duplicates.append(path)

master_df = master_df[~master_df['image_path'].isin(duplicates)].reset_index(drop=True)
print(f'  Removed {len(duplicates)} duplicate images')
print(f'\n✅ Clean dataset: {len(master_df)} images')

# Recompute class weights from actual frequencies
class_counts = master_df[CFG.DISEASES].sum().values
total = len(master_df)
computed_weights = total / (CFG.N_CLASSES * np.maximum(class_counts, 1))
computed_weights = computed_weights / computed_weights.min()  # normalize
print('\n⚖️  Computed class weights:')
for d, w in zip(CFG.DISEASES, computed_weights):
    print(f'  {d:5s}: {w:.3f}')
CFG.CLASS_WEIGHTS = computed_weights.tolist()

master_df.to_csv(CFG.OUTPUT_DIR / 'master_labels_clean.csv', index=False)
print('✅ master_labels_clean.csv saved')

## 🖼️ Stage 5 — Dataset Class & Augmentation

In [ ]:
def get_transforms(phase):
    if phase == 'train':
        return A.Compose([
            A.Resize(CFG.IMG_SIZE, CFG.IMG_SIZE),
            A.CLAHE(clip_limit=4.0, tile_grid_size=(8,8), p=0.5),
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.5),
            A.Rotate(limit=20, p=0.5),
            A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.15, rotate_limit=15, p=0.5),
            A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
            A.GaussNoise(var_limit=(10.0, 50.0), p=0.3),
            A.CoarseDropout(max_holes=8, max_height=32, max_width=32, p=0.3),
            A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
            ToTensorV2()
        ])
    else:
        return A.Compose([
            A.Resize(CFG.IMG_SIZE, CFG.IMG_SIZE),
            A.CLAHE(clip_limit=4.0, tile_grid_size=(8,8), p=1.0),
            A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
            ToTensorV2()
        ])

def apply_green_channel_enhance(img):
    """Ben Graham preprocessing — enhance green channel for retinal vessels"""
    img = cv2.addWeighted(img, 4, cv2.GaussianBlur(img, (0,0), sigmaX=10), -4, 128)
    return img

class RetinalDataset(Dataset):
    def __init__(self, df, transform=None, use_green_enhance=True):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.use_green_enhance = use_green_enhance

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = cv2.imread(row['image_path'])
        if img is None:
            img = np.zeros((CFG.IMG_SIZE, CFG.IMG_SIZE, 3), dtype=np.uint8)
        else:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            if self.use_green_enhance:
                img = apply_green_channel_enhance(img)

        if self.transform:
            img = self.transform(image=img)['image']

        label = torch.tensor(row[CFG.DISEASES].values.astype(np.float32))
        return img, label

print('✅ Dataset class & transforms ready')

# ─── Preview a few samples ───────────────────────────────────
sample_df = master_df.sample(4, random_state=42)
ds_preview = RetinalDataset(sample_df, transform=get_transforms('val'))
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for i, (img_t, lbl) in enumerate(ds_preview):
    img_np = img_t.permute(1,2,0).numpy()
    img_np = (img_np - img_np.min()) / (img_np.max() - img_np.min() + 1e-8)
    axes[i].imshow(img_np)
    diseases = [CFG.DISEASES[j] for j in range(CFG.N_CLASSES) if lbl[j] == 1]
    axes[i].set_title(', '.join(diseases) if diseases else 'Normal', fontsize=9)
    axes[i].axis('off')
plt.suptitle('Sample Images', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(CFG.OUTPUT_DIR / 'sample_images.png', dpi=100)
plt.show()
print('✅ Sample preview saved')

## 🏗️ Stage 6 — Model Architecture

In [ ]:
class EyeQModel(nn.Module):
    def __init__(self, model_name=CFG.MODEL_NAME, n_classes=CFG.N_CLASSES, pretrained=True, dropout=0.2):
        super().__init__()
        self.backbone = timm.create_model(model_name, pretrained=pretrained, num_classes=0)
        in_features = self.backbone.num_features
        self.head = nn.Sequential(
            nn.LayerNorm(in_features, eps=1e-6),
            nn.Identity(),
            nn.Linear(in_features, 512),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(512, n_classes)
        )

    def forward(self, x):
        features = self.backbone(x)
        return self.head(features)

# Test model
model_test = EyeQModel()
dummy = torch.randn(2, 3, CFG.IMG_SIZE, CFG.IMG_SIZE)
out = model_test(dummy)
total_params = sum(p.numel() for p in model_test.parameters()) / 1e6
print(f'✅ Model: ConvNeXt-Base')
print(f'   Input  : {list(dummy.shape)}')
print(f'   Output : {list(out.shape)}')
print(f'   Params : {total_params:.1f}M')
del model_test, dummy, out; gc.collect()

## ⚡ Stage 7 — Training Engine

In [ ]:
class Trainer:
    def __init__(self, model, optimizer, scheduler, criterion, scaler, device):
        self.model = model
        self.optimizer = optimizer
        self.scheduler = scheduler
        self.criterion = criterion
        self.scaler = scaler
        self.device = device

    def train_epoch(self, loader):
        self.model.train()
        total_loss = 0
        all_preds, all_labels = [], []

        for imgs, labels in tqdm(loader, desc='Train', leave=False):
            imgs   = imgs.to(self.device)
            labels = labels.to(self.device)

            self.optimizer.zero_grad()
            with autocast():
                logits = self.model(imgs)
                loss   = self.criterion(logits, labels)

            self.scaler.scale(loss).backward()
            self.scaler.unscale_(self.optimizer)
            nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
            self.scaler.step(self.optimizer)
            self.scaler.update()

            total_loss += loss.item()
            preds = torch.sigmoid(logits).detach().cpu().numpy()
            all_preds.append(preds)
            all_labels.append(labels.cpu().numpy())

        self.scheduler.step()
        preds_arr  = np.vstack(all_preds)
        labels_arr = np.vstack(all_labels)
        f1 = f1_score(labels_arr, (preds_arr > 0.5).astype(int),
                      average='macro', zero_division=0)
        return total_loss / len(loader), f1

    @torch.no_grad()
    def val_epoch(self, loader):
        self.model.eval()
        total_loss = 0
        all_preds, all_labels = [], []

        for imgs, labels in tqdm(loader, desc='Val  ', leave=False):
            imgs   = imgs.to(self.device)
            labels = labels.to(self.device)
            with autocast():
                logits = self.model(imgs)
                loss   = self.criterion(logits, labels)
            total_loss += loss.item()
            preds = torch.sigmoid(logits).cpu().numpy()
            all_preds.append(preds)
            all_labels.append(labels.cpu().numpy())

        preds_arr  = np.vstack(all_preds)
        labels_arr = np.vstack(all_labels)
        f1 = f1_score(labels_arr, (preds_arr > 0.5).astype(int),
                      average='macro', zero_division=0)
        try:
            auc = roc_auc_score(labels_arr, preds_arr, average='macro')
        except:
            auc = 0.0
        return total_loss / len(loader), f1, auc, preds_arr, labels_arr

def tune_thresholds(preds, labels):
    """Find optimal threshold per class to maximize F1"""
    best_thresholds = []
    for i in range(preds.shape[1]):
        best_t, best_f1 = 0.5, 0.0
        for t in np.arange(0.2, 0.8, 0.05):
            f1 = f1_score(labels[:,i], (preds[:,i] > t).astype(int), zero_division=0)
            if f1 > best_f1:
                best_f1 = f1
                best_t = t
        best_thresholds.append(round(best_t, 2))
    return best_thresholds

print('✅ Training engine ready')

## 🏋️ Stage 8 — K-Fold Training (Main Loop)

In [ ]:
# Load clean master CSV
master_df = pd.read_csv(CFG.OUTPUT_DIR / 'master_labels_clean.csv')
X = master_df['image_path'].values
Y = master_df[CFG.DISEASES].values

# ─── Multilabel Stratified K-Fold ────────────────────────────
mskf = MultilabelStratifiedKFold(n_splits=CFG.N_FOLDS, shuffle=True, random_state=CFG.SEED)

# ─── Loss function with class weights ────────────────────────
pos_weights = torch.tensor(CFG.CLASS_WEIGHTS, dtype=torch.float32).to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weights)

fold_results = []
oof_preds    = np.zeros((len(master_df), CFG.N_CLASSES))
oof_labels   = np.zeros((len(master_df), CFG.N_CLASSES))
all_thresholds = []

for fold, (train_idx, val_idx) in enumerate(mskf.split(X, Y)):
    print(f'\n{"="*60}')
    print(f'  FOLD {fold+1}/{CFG.N_FOLDS}')
    print(f'  Train: {len(train_idx)} | Val: {len(val_idx)}')
    print(f'{"="*60}')

    train_df = master_df.iloc[train_idx].reset_index(drop=True)
    val_df   = master_df.iloc[val_idx].reset_index(drop=True)

    train_ds = RetinalDataset(train_df, transform=get_transforms('train'))
    val_ds   = RetinalDataset(val_df,   transform=get_transforms('val'))

    train_loader = DataLoader(train_ds, batch_size=CFG.BATCH_SIZE,
                              shuffle=True,  num_workers=CFG.NUM_WORKERS,
                              pin_memory=True, drop_last=True)
    val_loader   = DataLoader(val_ds,   batch_size=CFG.BATCH_SIZE,
                              shuffle=False, num_workers=CFG.NUM_WORKERS,
                              pin_memory=True)

    # ─── Model ──────────────────────────────────────────────
    model = EyeQModel(pretrained=True)
    # Phase 1: Freeze backbone, train head only (5 warm-up epochs)
    for param in model.backbone.parameters():
        param.requires_grad = False

    if n_gpus > 1:
        model = nn.DataParallel(model)
    model = model.to(device)

    # ─── Optimizer & Scheduler ──────────────────────────────
    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=CFG.LR, weight_decay=CFG.WEIGHT_DECAY
    )
    scaler    = GradScaler()
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=CFG.EPOCHS, eta_min=1e-6
    )
    trainer = Trainer(model, optimizer, scheduler, criterion, scaler, device)

    best_val_f1   = 0.0
    best_model_path = CFG.MODEL_DIR / f'fold{fold+1}_best.pth'
    history = {'train_loss':[], 'val_loss':[], 'train_f1':[], 'val_f1':[], 'val_auc':[]}

    WARMUP_EPOCHS = 5

    for epoch in range(CFG.EPOCHS):
        # Unfreeze backbone after warmup
        if epoch == WARMUP_EPOCHS:
            print('  🔓 Unfreezing backbone...')
            core = model.module if n_gpus > 1 else model
            for param in core.backbone.parameters():
                param.requires_grad = True
            optimizer = torch.optim.AdamW(
                model.parameters(), lr=CFG.LR/5, weight_decay=CFG.WEIGHT_DECAY
            )
            scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
                optimizer, T_max=CFG.EPOCHS - WARMUP_EPOCHS, eta_min=1e-6
            )
            trainer.optimizer  = optimizer
            trainer.scheduler  = scheduler

        tr_loss, tr_f1 = trainer.train_epoch(train_loader)
        vl_loss, vl_f1, vl_auc, vl_preds, vl_labels = trainer.val_epoch(val_loader)

        history['train_loss'].append(tr_loss)
        history['val_loss'].append(vl_loss)
        history['train_f1'].append(tr_f1)
        history['val_f1'].append(vl_f1)
        history['val_auc'].append(vl_auc)

        print(f'  Epoch {epoch+1:02d}/{CFG.EPOCHS} | '
              f'TrLoss={tr_loss:.4f} TrF1={tr_f1:.4f} | '
              f'VlLoss={vl_loss:.4f} VlF1={vl_f1:.4f} VlAUC={vl_auc:.4f}',
              '⭐' if vl_f1 > best_val_f1 else '')

        if vl_f1 > best_val_f1:
            best_val_f1 = vl_f1
            best_val_preds  = vl_preds
            best_val_labels = vl_labels
            # Save: use model.module when DataParallel
            core = model.module if n_gpus > 1 else model
            torch.save(core.state_dict(), best_model_path)

    # Tune thresholds on best validation predictions
    thresholds = tune_thresholds(best_val_preds, best_val_labels)
    all_thresholds.append(thresholds)

    # Store OOF predictions
    oof_preds[val_idx]  = best_val_preds
    oof_labels[val_idx] = best_val_labels

    fold_results.append({'fold': fold+1, 'best_val_f1': best_val_f1,
                          'thresholds': thresholds})

    # Plot training curves
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(history['train_loss'], label='Train Loss')
    axes[0].plot(history['val_loss'],   label='Val Loss')
    axes[0].set_title(f'Fold {fold+1} — Loss'); axes[0].legend()
    axes[1].plot(history['train_f1'], label='Train F1')
    axes[1].plot(history['val_f1'],   label='Val F1')
    axes[1].set_title(f'Fold {fold+1} — Macro F1'); axes[1].legend()
    plt.tight_layout()
    plt.savefig(CFG.OUTPUT_DIR / f'fold{fold+1}_curves.png', dpi=100)
    plt.show()

    print(f'\n  ✅ Fold {fold+1} best val F1: {best_val_f1:.4f}')
    print(f'  📐 Tuned thresholds: {dict(zip(CFG.DISEASES, thresholds))}')

    del model, optimizer, scheduler, scaler, trainer
    del train_ds, val_ds, train_loader, val_loader
    gc.collect(); torch.cuda.empty_cache()

# ─── Overall OOF Score ───────────────────────────────────────
mean_thresholds = np.mean(all_thresholds, axis=0).tolist()
oof_binary = (oof_preds > np.array(mean_thresholds)).astype(int)
oof_f1 = f1_score(oof_labels, oof_binary, average='macro', zero_division=0)
print(f'\n🏆 Overall OOF Macro F1: {oof_f1:.4f}')
print(f'   Mean thresholds: {dict(zip(CFG.DISEASES, [round(t,2) for t in mean_thresholds]))}')

# Per-class metrics
print('\n📊 Per-class metrics (OOF):')
for i, d in enumerate(CFG.DISEASES):
    p  = precision_score(oof_labels[:,i], oof_binary[:,i], zero_division=0)
    r  = recall_score(oof_labels[:,i],    oof_binary[:,i], zero_division=0)
    f  = f1_score(oof_labels[:,i],        oof_binary[:,i], zero_division=0)
    print(f'  {d:5s}: P={p:.3f} R={r:.3f} F1={f:.3f}')

# Load clean master CSV
master_df = pd.read_csv(CFG.OUTPUT_DIR / 'master_labels_clean.csv')
X = master_df['image_path'].values
Y = master_df[CFG.DISEASES].values

# ─── Multilabel Stratified K-Fold ────────────────────────────
mskf = MultilabelStratifiedKFold(n_splits=CFG.N_FOLDS, shuffle=True, random_state=CFG.SEED)

# ─── Loss function with class weights ────────────────────────
pos_weights = torch.tensor(CFG.CLASS_WEIGHTS, dtype=torch.float32).to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weights)

fold_results = []
oof_preds    = np.zeros((len(master_df), CFG.N_CLASSES))
oof_labels   = np.zeros((len(master_df), CFG.N_CLASSES))
oof_mask     = np.zeros(len(master_df), dtype=bool)
all_thresholds = []

for fold, (train_idx, val_idx) in enumerate(mskf.split(X, Y)):
    if fold >= CFG.LIMIT_FOLDS:
        print(f'\n⏭️ Skipping Fold {fold+1} due to LIMIT_FOLDS={CFG.LIMIT_FOLDS} limitation')
        break

    print(f'\n{"="*60}')
    print(f'  FOLD {fold+1}/{CFG.N_FOLDS}')
    print(f'  Train: {len(train_idx)} | Val: {len(val_idx)}')
    print(f'{"="*60}')

    train_df = master_df.iloc[train_idx].reset_index(drop=True)
    val_df   = master_df.iloc[val_idx].reset_index(drop=True)

    train_ds = RetinalDataset(train_df, transform=get_transforms('train'))
    val_ds   = RetinalDataset(val_df,   transform=get_transforms('val'))

    train_loader = DataLoader(train_ds, batch_size=CFG.BATCH_SIZE,
                              shuffle=True,  num_workers=CFG.NUM_WORKERS,
                              pin_memory=True, drop_last=True)
    val_loader   = DataLoader(val_ds,   batch_size=CFG.BATCH_SIZE,
                              shuffle=False, num_workers=CFG.NUM_WORKERS,
                              pin_memory=True)

    # ─── Model ──────────────────────────────────────────────
    model = EyeQModel(pretrained=True)
    # Phase 1: Freeze backbone, train head only
    for param in model.backbone.parameters():
        param.requires_grad = False

    if n_gpus > 1:
        model = nn.DataParallel(model)
    model = model.to(device)

    # ─── Optimizer & Scheduler ──────────────────────────────
    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=CFG.LR, weight_decay=CFG.WEIGHT_DECAY
    )
    scaler    = GradScaler()
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=CFG.EPOCHS, eta_min=1e-6
    )
    trainer = Trainer(model, optimizer, scheduler, criterion, scaler, device)

    best_val_f1   = 0.0
    best_model_path = CFG.MODEL_DIR / f'fold{fold+1}_best.pth'
    history = {'train_loss':[], 'val_loss':[], 'train_f1':[], 'val_f1':[], 'val_auc':[]}

    # Dynamically set warmup epochs based on total epochs
    WARMUP_EPOCHS = 1 if CFG.EPOCHS <= 2 else max(1, CFG.EPOCHS // 6)

    for epoch in range(CFG.EPOCHS):
        # Unfreeze backbone after warmup
        if epoch == WARMUP_EPOCHS:
            print('  🔓 Unfreezing backbone...')
            core = model.module if n_gpus > 1 else model
            for param in core.backbone.parameters():
                param.requires_grad = True
            optimizer = torch.optim.AdamW(
                model.parameters(), lr=CFG.LR/5, weight_decay=CFG.WEIGHT_DECAY
            )
            scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
                optimizer, T_max=CFG.EPOCHS - WARMUP_EPOCHS, eta_min=1e-6
            )
            trainer.optimizer  = optimizer
            trainer.scheduler  = scheduler

        tr_loss, tr_f1 = trainer.train_epoch(train_loader)
        vl_loss, vl_f1, vl_auc, vl_preds, vl_labels = trainer.val_epoch(val_loader)

        history['train_loss'].append(tr_loss)
        history['val_loss'].append(vl_loss)
        history['train_f1'].append(tr_f1)
        history['val_f1'].append(vl_f1)
        history['val_auc'].append(vl_auc)

        print(f'  Epoch {epoch+1:02d}/{CFG.EPOCHS} | '
              f'TrLoss={tr_loss:.4f} TrF1={tr_f1:.4f} | '
              f'VlLoss={vl_loss:.4f} VlF1={vl_f1:.4f} VlAUC={vl_auc:.4f}',
              '⭐' if vl_f1 > best_val_f1 else '')

        if vl_f1 > best_val_f1:
            best_val_f1 = vl_f1
            best_val_preds  = vl_preds
            best_val_labels = vl_labels
            # Save: use model.module when DataParallel
            core = model.module if n_gpus > 1 else model
            torch.save(core.state_dict(), best_model_path)

    # Tune thresholds on best validation predictions
    thresholds = tune_thresholds(best_val_preds, best_val_labels)
    all_thresholds.append(thresholds)

    # Store OOF predictions
    oof_preds[val_idx]  = best_val_preds
    oof_labels[val_idx] = best_val_labels
    oof_mask[val_idx]   = True

    fold_results.append({'fold': fold+1, 'best_val_f1': best_val_f1,
                          'thresholds': thresholds})

    # Plot training curves
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(history['train_loss'], label='Train Loss')
    axes[0].plot(history['val_loss'],   label='Val Loss')
    axes[0].set_title(f'Fold {fold+1} — Loss'); axes[0].legend()
    axes[1].plot(history['train_f1'], label='Train F1')
    axes[1].plot(history['val_f1'],   label='Val F1')
    axes[1].set_title(f'Fold {fold+1} — Macro F1'); axes[1].legend()
    plt.tight_layout()
    plt.savefig(CFG.OUTPUT_DIR / f'fold{fold+1}_curves.png', dpi=100)
    plt.show()

    print(f'\n  ✅ Fold {fold+1} best val F1: {best_val_f1:.4f}')
    print(f'  📐 Tuned thresholds: {dict(zip(CFG.DISEASES, thresholds))}')

    del model, optimizer, scheduler, scaler, trainer
    del train_ds, val_ds, train_loader, val_loader
    gc.collect(); torch.cuda.empty_cache()

# ─── Overall OOF Score ───────────────────────────────────────
if oof_mask.sum() > 0:
    valid_oof_preds = oof_preds[oof_mask]
    valid_oof_labels = oof_labels[oof_mask]
else:
    valid_oof_preds = oof_preds
    valid_oof_labels = oof_labels

mean_thresholds = np.mean(all_thresholds, axis=0).tolist()
oof_binary = (valid_oof_preds > np.array(mean_thresholds)).astype(int)
oof_f1 = f1_score(valid_oof_labels, oof_binary, average='macro', zero_division=0)
print(f'\n🏆 Overall OOF Macro F1: {oof_f1:.4f}')
print(f'   Mean thresholds: {dict(zip(CFG.DISEASES, [round(t,2) for t in mean_thresholds]))}')

# Per-class metrics
print('\n📊 Per-class metrics (OOF):')
for i, d in enumerate(CFG.DISEASES):
    p  = precision_score(valid_oof_labels[:,i], oof_binary[:,i], zero_division=0)
    r  = recall_score(valid_oof_labels[:,i],    oof_binary[:,i], zero_division=0)
    f  = f1_score(valid_oof_labels[:,i],        oof_binary[:,i], zero_division=0)
    print(f'  {d:5s}: P={p:.3f} R={r:.3f} F1={f:.3f}')

In [ ]:
# Pick best fold by val F1
best_fold_info = max(fold_results, key=lambda x: x['best_val_f1'])
best_fold      = best_fold_info['fold']
best_f1        = best_fold_info['best_val_f1']
print(f'🏆 Best fold: {best_fold} with Val F1 = {best_f1:.4f}')

# Copy best model to outputs
src = CFG.MODEL_DIR / f'fold{best_fold}_best.pth'
dst = CFG.OUTPUT_DIR / 'best_model.pth'
shutil.copy(src, dst)

# Save metadata
meta = {
    'best_fold'        : best_fold,
    'best_val_f1'      : best_f1,
    'oof_macro_f1'     : oof_f1,
    'mean_thresholds'  : dict(zip(CFG.DISEASES, [round(t,2) for t in mean_thresholds])),
    'model_name'       : CFG.MODEL_NAME,
    'img_size'         : CFG.IMG_SIZE,
    'diseases'         : CFG.DISEASES,
    'fold_results'     : fold_results,
}
with open(CFG.OUTPUT_DIR / 'training_meta.json', 'w') as f:
    json.dump(meta, f, indent=2)

print(f'✅ best_model.pth saved → {dst}')
print(f'✅ training_meta.json  saved')

# ─── Fold comparison bar chart ───────────────────────────────
folds = [r['fold'] for r in fold_results]
f1s   = [r['best_val_f1'] for r in fold_results]
colors = ['#2ecc71' if f == best_fold else '#3498db' for f in folds]
plt.figure(figsize=(8, 4))
bars = plt.bar([f'Fold {f}' for f in folds], f1s, color=colors, edgecolor='white')
for bar, val in zip(bars, f1s):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
             f'{val:.4f}', ha='center', va='bottom', fontsize=10)
plt.axhline(oof_f1, color='red', linestyle='--', label=f'OOF F1={oof_f1:.4f}')
plt.title('K-Fold Macro F1 Scores', fontsize=14, fontweight='bold')
plt.ylabel('Macro F1'); plt.legend(); plt.ylim(0, 1)
plt.tight_layout()
plt.savefig(CFG.OUTPUT_DIR / 'fold_comparison.png', dpi=100)
plt.show()

## 🔥 Stage 10 — GradCAM Explainability

In [ ]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

# Load best model
best_model = EyeQModel(pretrained=False).to(device)
best_model.load_state_dict(torch.load(CFG.OUTPUT_DIR / 'best_model.pth', map_location=device))
best_model.eval()

# Target layer = last stage of ConvNeXt
target_layer = [best_model.backbone.stages[-1].blocks[-1].conv_dw]

gradcam = GradCAM(model=best_model, target_layers=target_layer)

# Generate for 10 sample images (one per disease)
gradcam_dir = CFG.OUTPUT_DIR / 'gradcam'
gradcam_dir.mkdir(exist_ok=True)

transform_val = get_transforms('val')

print('Generating GradCAM visualizations...')
for disease_idx, disease_name in enumerate(CFG.DISEASES):
    # Find a sample with this disease
    samples = master_df[master_df[disease_name] == 1]
    if len(samples) == 0:
        print(f'  ⚠️  No sample for {disease_name}')
        continue
    sample = samples.sample(1, random_state=42).iloc[0]

    img_orig = cv2.cvtColor(cv2.imread(sample['image_path']), cv2.COLOR_BGR2RGB)
    img_orig = cv2.resize(img_orig, (CFG.IMG_SIZE, CFG.IMG_SIZE))
    img_float = img_orig.astype(np.float32) / 255.0

    img_t = transform_val(image=img_orig)['image'].unsqueeze(0).to(device)

    targets = [ClassifierOutputTarget(disease_idx)]
    cam = gradcam(input_tensor=img_t, targets=targets)
    cam_image = show_cam_on_image(img_float, cam[0], use_rgb=True)

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].imshow(img_orig);   axes[0].set_title('Original');           axes[0].axis('off')
    axes[1].imshow(cam_image);  axes[1].set_title(f'GradCAM: {disease_name}'); axes[1].axis('off')
    plt.suptitle(f'Disease: {disease_name}', fontweight='bold')
    plt.tight_layout()
    plt.savefig(gradcam_dir / f'gradcam_{disease_name}.png', dpi=100)
    plt.close()
    print(f'  ✅ {disease_name}')

print(f'\n✅ GradCAM images saved → {gradcam_dir}')

## 📄 Stage 11 — Retinal Health Index & PDF Report

In [ ]:
from reportlab.lib.pagesizes import A4
from reportlab.lib import colors
from reportlab.platypus import SimpleDocTemplate, Table, TableStyle, Paragraph, Spacer, Image as RLImage
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import inch

DISEASE_DESCRIPTIONS = {
    'DR' : 'Diabetic Retinopathy — damage to blood vessels caused by diabetes.',
    'G'  : 'Glaucoma — optic nerve damage, often from elevated eye pressure.',
    'AMD': 'Age-related Macular Degeneration — central vision deterioration.',
    'C'  : 'Cataract — clouding of the lens reducing visual clarity.',
    'M'  : 'Pathological Myopia — severe nearsightedness with retinal changes.',
    'HR' : 'Hypertensive Retinopathy — retinal damage due to high blood pressure.',
    'DME': 'Diabetic Macular Edema — fluid accumulation in the macula.',
    'P'  : 'Papilledema — optic disc swelling from increased intracranial pressure.',
    'CSR': 'Central Serous Retinopathy — fluid under the retina causing visual distortion.',
    'RVO': 'Retinal Vein Occlusion — blockage of retinal veins causing hemorrhage.',
}

def compute_rhi(predictions: dict) -> float:
    """Retinal Health Index: 0 (critical) to 100 (healthy)"""
    weights = {'DR':0.15,'G':0.15,'AMD':0.1,'C':0.08,'M':0.08,
               'HR':0.1,'DME':0.12,'P':0.1,'CSR':0.07,'RVO':0.05}
    risk_score = sum(predictions.get(d, 0) * w for d, w in weights.items())
    return round(max(0, 100 - risk_score * 100), 1)

def generate_pdf_report(patient_id, img_path, predictions, output_path):
    doc = SimpleDocTemplate(str(output_path), pagesize=A4,
                            leftMargin=0.75*inch, rightMargin=0.75*inch,
                            topMargin=0.75*inch, bottomMargin=0.75*inch)
    styles = getSampleStyleSheet()
    story  = []

    # Title
    title_style = ParagraphStyle('Title', fontSize=18, fontName='Helvetica-Bold',
                                  textColor=colors.HexColor('#1a237e'), spaceAfter=6)
    story.append(Paragraph('EyeQ Innovate — Retinal Analysis Report', title_style))
    story.append(Paragraph(f'Patient ID: {patient_id}', styles['Normal']))
    story.append(Spacer(1, 0.2*inch))

    # RHI
    rhi = compute_rhi(predictions)
    rhi_color = '#27ae60' if rhi > 70 else '#f39c12' if rhi > 40 else '#e74c3c'
    rhi_style = ParagraphStyle('RHI', fontSize=14, fontName='Helvetica-Bold',
                                textColor=colors.HexColor(rhi_color))
    story.append(Paragraph(f'Retinal Health Index (RHI): {rhi}/100', rhi_style))
    story.append(Spacer(1, 0.15*inch))

    # Predictions table
    story.append(Paragraph('Disease Predictions:', styles['Heading2']))
    table_data = [['Disease', 'Confidence', 'Status']]
    for disease, conf in predictions.items():
        status = '⚠️ Detected' if conf > 0.5 else '✅ Normal'
        table_data.append([DISEASE_DESCRIPTIONS[disease][:40]+'...'
                           if len(DISEASE_DESCRIPTIONS[disease]) > 40
                           else DISEASE_DESCRIPTIONS[disease],
                           f'{conf*100:.1f}%', status])
    t = Table(table_data, colWidths=[3.5*inch, 1*inch, 1.2*inch])
    t.setStyle(TableStyle([
        ('BACKGROUND',   (0,0), (-1,0), colors.HexColor('#1a237e')),
        ('TEXTCOLOR',    (0,0), (-1,0), colors.white),
        ('FONTNAME',     (0,0), (-1,0), 'Helvetica-Bold'),
        ('FONTSIZE',     (0,0), (-1,-1), 9),
        ('ROWBACKGROUNDS', (0,1), (-1,-1), [colors.white, colors.HexColor('#f5f5f5')]),
        ('GRID',         (0,0), (-1,-1), 0.5, colors.grey),
        ('ALIGN',        (1,0), (-1,-1), 'CENTER'),
        ('VALIGN',       (0,0), (-1,-1), 'MIDDLE'),
        ('PADDING',      (0,0), (-1,-1), 6),
    ]))
    story.append(t)
    story.append(Spacer(1, 0.2*inch))
    story.append(Paragraph('<b>Disclaimer:</b> This AI analysis is for screening purposes only. '
                            'Please consult an ophthalmologist for clinical diagnosis.',
                            styles['Normal']))

    doc.build(story)
    return output_path

# Demo report
demo_preds = {d: float(np.random.uniform(0.1, 0.9)) for d in CFG.DISEASES}
demo_preds['DR'] = 0.82; demo_preds['DME'] = 0.65
sample_img = master_df.sample(1).iloc[0]['image_path']
report_path = CFG.OUTPUT_DIR / 'demo_report.pdf'
generate_pdf_report('DEMO-001', sample_img, demo_preds, report_path)
print(f'✅ Demo PDF report saved → {report_path}')

## 📦 Stage 12 — Export ONNX Model

In [ ]:
import torch.onnx

best_model.eval()
dummy_input = torch.randn(1, 3, CFG.IMG_SIZE, CFG.IMG_SIZE).to(device)
onnx_path   = CFG.OUTPUT_DIR / 'eyeq_model.onnx'

torch.onnx.export(
    best_model,
    dummy_input,
    str(onnx_path),
    export_params    = True,
    opset_version    = 17,
    do_constant_folding = True,
    input_names      = ['retinal_image'],
    output_names     = ['disease_logits'],
    dynamic_axes     = {
        'retinal_image'  : {0: 'batch_size'},
        'disease_logits' : {0: 'batch_size'}
    }
)
print(f'✅ ONNX model exported → {onnx_path}')

# Verify ONNX
try:
    import onnx
    onnx_model = onnx.load(str(onnx_path))
    onnx.checker.check_model(onnx_model)
    print('✅ ONNX model verified successfully')
except ImportError:
    print('ℹ️  onnx package not installed — skipping verification')

## 📊 Stage 13 — Final Summary & Output Files

In [ ]:
print('='*60)
print(' 🔬 EyeQ INNOVATE — TRAINING COMPLETE')
print('='*60)

with open(CFG.OUTPUT_DIR / 'training_meta.json') as f:
    meta = json.load(f)

print(f"\n  Model         : {meta['model_name']}")
print(f"  Best Fold     : {meta['best_fold']}")
print(f"  Best Val F1   : {meta['best_val_f1']:.4f}")
print(f"  OOF Macro F1  : {meta['oof_macro_f1']:.4f}")
print(f"  Thresholds    : {meta['mean_thresholds']}")

print('\n📂 Output Files:')
for f in sorted(CFG.OUTPUT_DIR.rglob('*')):
    if f.is_file():
        size_kb = f.stat().st_size / 1024
        print(f'  {f.name:40s} {size_kb:8.1f} KB')

print('\n✅ Download best_model.pth and eyeq_model.onnx from /kaggle/working/outputs/')
print('✅ Use training_meta.json to restore thresholds at inference time')